# Device Lookup Batch — Bronze → Silver Cleaning

Cleans the `device_lookup_batch` bronze data and publishes the Silver table.

**Pipeline**
1. **Data Cleaning** — parse `notes`, standardize timestamps/dates, project to the 20-column Silver schema, dedupe, and stage to Parquet.
2. **Data Validation** — reload the staged Parquet and run business-contract checks (FAIL counts double as data-quality KPIs for carrier reporting).
3. **Data Checking** — drill into specific data-quality problems with bronze-side views.
4. **Save Data** — publish to the partitioned Iceberg Silver table.

**Input:** pre-decompressed bronze at `s3://enstream-lake-silver-dev/temp_raw/device_lookup_batch/`  
**Output:** staged Parquet + Iceberg table `trust_score_v1_poc_silver.device_lookup_batch`

Unit / invariant tests for the cleaning functions live in the companion notebook `device_lookup_batch_tests_v6.ipynb`.

In [ ]:
%%pyspark project.spark.compatibility
import boto3

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType

In [ ]:
%%pyspark project.spark.compatibility
spark = (
    SparkSession.builder
    .appName("device_lookup_batch")
    .config("spark.executor.memory", "18g")
    .config("spark.driver.memory", "10g")
    .config("spark.executor.cores", "4")
    .config("spark.sql.shuffle.partitions", "300")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .getOrCreate()
)

In [ ]:
%%pyspark project.spark.compatibility
# Bronze `timestamp` is already in America/Toronto wall-clock time (EST/EDT).
# Set session tz to match so to_timestamp preserves the wall-clock string on parse.
spark.conf.set("spark.sql.session.timeZone", "America/Toronto")

In [ ]:
%%pyspark project.spark.compatibility
# -------------------------------------------------------------------------
# Original bronze read (kept for reference; very slow due to gzip on read).
# One-time prep was: decompress + write to temp_raw parquet (cell below).
# -------------------------------------------------------------------------
# s3_path = "s3://enstream-lake-bronze-dev/landing/device_lookup_batch.csv.gz"
# dlb = spark.read.option("header", True).csv(s3_path)
# dlb.printSchema()
# dlb.write.mode("overwrite").parquet("s3://enstream-lake-silver-dev/temp_raw/device_lookup_batch/")

In [ ]:
%%pyspark project.spark.compatibility
# Read the pre-decompressed bronze copy (much faster than re-reading .csv.gz)
dlb = spark.read.parquet("s3://enstream-lake-silver-dev/temp_raw/device_lookup_batch/")
dlb.printSchema()
dlb.show(5, truncate=False)

In [ ]:
%%pyspark project.spark.compatibility
dlb.cache()
dlb.count()

# Data Cleaning

Parse the `notes` JSON, standardize timestamps/dates, project to the 20-column Silver schema, dedupe, and stage to Parquet.

In [ ]:
%%pyspark project.spark.compatibility
def profile_json_keys(
    df: DataFrame,
    json_col: str,
    treat_backslash_n_as_null: bool = True,
    empty_as_null: bool = True,
) -> DataFrame:
    """List all keys appearing in a JSON-string column with their occurrence counts."""
    condition = F.col(json_col).isNotNull()
    if treat_backslash_n_as_null:
        condition = condition & (F.col(json_col) != "\\N")
    if empty_as_null:
        condition = condition & (F.trim(F.col(json_col)) != "")

    return (
        df.select(json_col)
          .filter(condition)
          .withColumn("json_map", F.from_json(F.col(json_col), "map<string,string>"))
          .select(F.explode(F.map_keys(F.col("json_map"))).alias("json_key"))
          .groupBy("json_key").count()
          .orderBy(F.desc("count"))
    )

In [ ]:
%%pyspark project.spark.compatibility
profile_json_keys(dlb, "notes")

In [ ]:
%%pyspark project.spark.compatibility
def flatten_json_column_with_schema(
    df: DataFrame,
    json_col: str,
    schema: StructType,
    prefix: str = None,
    drop_original: bool = False,
) -> DataFrame:
    """Parse a JSON-string column with a given schema and promote each field to its own column."""
    parsed = F.from_json(F.col(json_col), schema)
    for field in schema.fieldNames():
        output_col = f"{prefix}_{field}" if prefix else field
        df = df.withColumn(output_col, parsed[field])
    if drop_original:
        df = df.drop(json_col)
    return df

In [ ]:
%%pyspark project.spark.compatibility
# notes JSON schema for device_lookup_batch.
# correlationId may not appear in every row; will be null where absent.
notes_schema = StructType([
    StructField("tac",             StringType(), True),
    StructField("oldIMSI",         StringType(), True),
    StructField("oldIMEI",         StringType(), True),
    StructField("fromADCSnapshot", StringType(), True),
    StructField("correlationId",   StringType(), True),
])

In [ ]:
%%pyspark project.spark.compatibility
dlb_parsed = flatten_json_column_with_schema(
    df=dlb,
    json_col="notes",
    schema=notes_schema,
    prefix=None,
    drop_original=True,
)

In [ ]:
%%pyspark project.spark.compatibility
def is_blank(c):
    """Return True when `c` is null, "\\N", empty, whitespace-only, or "MISSING_DATA"."""
    t = F.trim(F.col(c))
    return F.col(c).isNull() | (t == "\\N") | (t == "") | (t == "MISSING_DATA")

In [ ]:
%%pyspark project.spark.compatibility
def standardize_timestamp(df, ts_col, output_col=None):
    """Parse a string timestamp (yyyy-MM-dd HH:mm:ss or yyyy-MM-dd) into TimestampType; unparseable -> null. Wall-clock time is interpreted in the session timezone (set to America/Toronto)."""
    target_col = output_col or ts_col
    parsed = F.coalesce(
        F.try_to_timestamp(F.col(ts_col), F.lit("yyyy-MM-dd HH:mm:ss")),
        F.try_to_timestamp(F.col(ts_col), F.lit("yyyy-MM-dd")),
    )
    return df.withColumn(target_col, parsed)

def standardize_date(df, date_col, output_col=None):
    """Parse a string date column (yyyy-MM-dd) into DateType; unparseable -> null.
    Pre-filters to the yyyy-MM-dd pattern before to_date so bad input becomes null
    under any Spark version (no try_to_date dependency)."""
    target_col = output_col or date_col
    s = F.col(date_col)
    safe = F.when(s.rlike(r"^\d{4}-\d{2}-\d{2}$"), s).otherwise(None)
    return df.withColumn(target_col, F.to_date(safe, "yyyy-MM-dd"))

In [ ]:
%%pyspark project.spark.compatibility
dlb_ts    = standardize_timestamp(dlb_parsed, "timestamp")
dlb_dated = standardize_date(dlb_ts, "date")

In [ ]:
%%pyspark project.spark.compatibility
# Project bronze -> silver schema (20 columns)

def clean_str(c):
    """Trim; convert literal "\\N", empty string, and "MISSING_DATA"
    (Rogers placeholder for unknown values) to null."""
    v = F.trim(F.col(c))
    return F.when((v == "\\N") | (v == "") | (v == "MISSING_DATA"), None).otherwise(v)

def safe_long(c):
    """Cast a string column to bigint. Pure digits only; non-numeric -> null.
    LongType required: bronze `id` is bigint and values can exceed Int32 max."""
    s = clean_str(c)
    return F.when(s.rlike("^[0-9]+$"), s).otherwise(None).cast(LongType())

def to_silver(df):
    """Project the dlb frame onto the silver schema (20 columns).
    `event_date` is derived from `event_timestamp` so the partition column is
    always internally consistent with the event time; bronze `date` is kept
    alongside for source-fidelity tracking."""
    return df.select(
        safe_long("id").alias("record_id"),
        clean_str("change_key").alias("phone_number_AC_hash"),
        clean_str("change_key_ats").alias("phone_number_AT_hash"),
        F.upper(clean_str("mno")).alias("mno"),
        clean_str("imei").alias("imei"),
        clean_str("imsi").alias("imsi"),
        F.col("date"),                                                       # bronze date kept
        F.col("timestamp").alias("event_timestamp"),
        clean_str("correlationId").alias("correlation_id"),
        clean_str("related_id").alias("related_id"),
        F.upper(clean_str("status")).alias("event_type"),
        clean_str("tac").alias("tac"),
        clean_str("oldIMSI").alias("oldIMSI"),
        clean_str("oldIMEI").alias("oldIMEI"),
        clean_str("fromADCSnapshot").alias("fromADCSnapshot"),
        F.to_timestamp(F.lit("2026-03-16 00:00:00")).alias("ingestion_ts"),
        F.lit("MYSQL").alias("source_name"),
        F.lit(None).cast(LongType()).alias("source_event_id"),
        F.lit(1).cast(IntegerType()).alias("schema_version"),
        F.to_date(F.col("timestamp")).alias("event_date"),                   # NEW: derived, partition col
    )

dlb_final = to_silver(dlb_dated)

In [ ]:
%%pyspark project.spark.compatibility
# Drop fully-identical rows
dlb_final = dlb_final.dropDuplicates()

In [ ]:
%%pyspark project.spark.compatibility
dlb_final.show(1, False, True)

In [ ]:
%%pyspark project.spark.compatibility
dlb_final.printSchema()

In [ ]:
%%pyspark project.spark.compatibility
# Persist cleaned silver to a temp parquet location so downstream cells
# (validation, EDA, tests) read from disk instead of re-running the full
# cleaning lineage on every query.
# Mirror of the bronze temp_raw pattern in cell 3-4.
#
# Run this ONCE after any change to the cleaning pipeline. On subsequent
# notebook runs where cleaning is unchanged, you can skip this cell and
# go straight to the validation section (which reads from this path).
dlb_final.write.mode("overwrite").parquet(
    "s3://enstream-lake-silver-dev/temp_raw/device_lookup_batch_cleaned/"
)

# Data Validation

Reload the staged Parquet and assert the business contract on the real data. FAIL counts are intentional KPIs that surface missing/invalid data to report back to carriers.

In [ ]:
%%pyspark project.spark.compatibility
# Read the cleaned silver from temp_raw (written by the last cell of the cleaning section).
# This breaks the lineage to the bronze read + cleaning chain, so downstream
# validation / EDA queries are single parquet scans instead of full pipeline re-execution.
dlb_silver = spark.read.parquet(
    "s3://enstream-lake-silver-dev/temp_raw/device_lookup_batch_cleaned/"
)

In [ ]:
%%pyspark project.spark.compatibility
# dlb_silver.cache()
# dlb_silver.count()

In [ ]:
%%pyspark project.spark.compatibility
dlb_orig_nulls = dlb.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in dlb.columns
])
dlb_orig_nulls.show()


In [ ]:
%%pyspark project.spark.compatibility
# Null counts per column. MISSING_DATA is already converted to null in to_silver
# (treated identically to \N), so a plain isNull() check is enough.
dlb_nulls = dlb_silver.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in dlb_silver.columns
])
dlb_nulls.show()

In [ ]:
%%pyspark project.spark.compatibility
# +---+----------+--------------+---+--------+--------+----+---------+--------+----------+------+
# | id|change_key|change_key_ats|mno|    imei|    imsi|date|timestamp|   notes|related_id|status|
# +---+----------+--------------+---+--------+--------+----+---------+--------+----------+------+
# |  0|         0|             0|  0|32785670|67637768|   0|        0|28789505|         0|     0|
# +---+----------+--------------+---+--------+--------+----+---------+--------+----------+------+
# +---------+--------------------+--------------------+---+--------+--------+----+---------------+--------------+----------+----------+--------+---------+---------+---------------+------------+-----------+---------------+--------------+
# |record_id|phone_number_AC_hash|phone_number_AT_hash|mno|    imei|    imsi|date|event_timestamp|correlation_id|related_id|event_type|     tac|  oldIMSI|  oldIMEI|fromADCSnapshot|ingestion_ts|source_name|source_event_id|schema_version|
# +---------+--------------------+--------------------+---+--------+--------+----+---------------+--------------+----------+----------+--------+---------+---------+---------------+------------+-----------+---------------+--------------+
# |        0|                   0|            20971673|  0|41043350|75895448|   0|              0|     412347877|         0|         0|41043407|400619036|362563067|      406627781|           0|          0|      412347877|             0|
# +---------+--------------------+--------------------+---+--------+--------+----+---------------+--------------+----------+----------+--------+---------+---------+---------------+------------+-----------+---------------+--------------+


In [ ]:
%%pyspark project.spark.compatibility
dlb_silver.show(10, False)

In [ ]:
%%pyspark project.spark.compatibility
# ============================================================
# Data validation on dlb_silver - business-contract checks
# Sources:
#   - Image 1 (bronze schema): NOT NULL columns
#   - Image 2: mno -> allowed status (event_type) matrix
#   - Helena's note: all nulls are unacceptable; FAIL counts are KPIs to surface
#     missing data with carriers (especially for the future graph database).
# Each check counts "rows that violate the contract"; expected = 0.
# ============================================================

ALLOWED_EVENT_TYPES_BY_MNO = {
    "BELL":   {"IMEI_END", "IMEI_START", "IMSI_END", "IMSI_START"},
    "TELUS":  {"IMEI_END", "IMEI_START", "IMEI_START_A", "IMEI_START_NC",
               "IMSI_END_C", "IMSI_END", "IMSI_START", "IMSI_START_A", "IMSI_START_NC"},
    "ROGERS": {"IMEI_END", "IMEI_START", "IMSI_END", "IMSI_START"},
}
ALL_EVENT_TYPES = set().union(*ALLOWED_EVENT_TYPES_BY_MNO.values())
HEX64 = "(?i)^[0-9a-f]{64}$"

def _check(label, violation_cond):
    """Count rows that violate the contract; print PASS / FAIL with the count."""
    n = dlb_silver.filter(violation_cond).count()
    status = "PASS" if n == 0 else f"FAIL ({n})"
    print(f"{status:<14} {label}")
    return n

print("---- dlb_silver data validation ----")
print(f"total rows: {dlb_silver.count()}\n")

# 1. NOT NULL coverage (Helena: any null is a data-quality issue).
#    Skip phase-1 intentional null + notes-parsed optional fields.
EXPECTED_NULL_COLUMNS = {
    "source_event_id",
    "tac", "oldIMSI", "oldIMEI", "fromADCSnapshot", "correlation_id",
}
print("--- NOT NULL coverage ---")
for c in dlb_silver.columns:
    if c in EXPECTED_NULL_COLUMNS:
        continue
    _check(f"{c} IS NOT NULL", F.col(c).isNull())

# 2. mno domain
_check("mno IN (BELL, TELUS, ROGERS)",
       ~F.col("mno").isin(["BELL", "TELUS", "ROGERS"]))

# 3. event_type in the known set
_check(f"event_type IN known set (n={len(ALL_EVENT_TYPES)})",
       ~F.col("event_type").isin(list(ALL_EVENT_TYPES)))

# 4. (mno, event_type) pairs follow Image 2
for mno, allowed in ALLOWED_EVENT_TYPES_BY_MNO.items():
    _check(f"{mno}.event_type subset of allowed list",
           (F.col("mno") == mno) & ~F.col("event_type").isin(list(allowed)))

# 5. Hash format checks (64-char hex)
_check("phone_number_AC_hash matches 64-char hex",
       ~F.col("phone_number_AC_hash").rlike(HEX64))
_check("phone_number_AT_hash (when not null) matches 64-char hex",
       F.col("phone_number_AT_hash").isNotNull() & ~F.col("phone_number_AT_hash").rlike(HEX64))
_check("imei (when not null) matches 64-char hex",
       F.col("imei").isNotNull() & ~F.col("imei").rlike(HEX64))
_check("imsi (when not null) matches 64-char hex",
       F.col("imsi").isNotNull() & ~F.col("imsi").rlike(HEX64))
_check("oldIMEI (when not null) matches 64-char hex",
       F.col("oldIMEI").isNotNull() & ~F.col("oldIMEI").rlike(HEX64))
_check("oldIMSI (when not null) matches 64-char hex",
       F.col("oldIMSI").isNotNull() & ~F.col("oldIMSI").rlike(HEX64))

# 6. tac is 8 digits when not null
_check("tac (when not null) matches 8 digits",
       F.col("tac").isNotNull() & ~F.col("tac").rlike(r"^\d{8}$"))

# 7. related_id (when not null) is numeric
_check("related_id (when not null) is numeric",
       F.col("related_id").isNotNull() & ~F.col("related_id").rlike(r"^\d+$"))

# 8. record_id > 0 (bronze: bigint NOT NULL)
_check("record_id > 0", F.col("record_id") <= 0)

# 9. IMEI_START events should carry a non-null imei
_check("IMEI_START => imei IS NOT NULL",
       (F.col("event_type") == "IMEI_START") & F.col("imei").isNull())

# 10. IMSI_START events should carry a non-null imsi
_check("IMSI_START => imsi IS NOT NULL",
       (F.col("event_type") == "IMSI_START") & F.col("imsi").isNull())

# 11. bronze `date` should agree with derived `event_date` (sanity check on source consistency)
_check("date == event_date (source-vs-derived agreement)",
       F.col("date").isNotNull()
       & F.col("event_date").isNotNull()
       & (F.col("date") != F.col("event_date")))

# Data Checking (specific problem)

Drill into specific data-quality issues (literal `\N`, non-hex hash IDs, non-8-digit `tac`) with bronze-side views, to take upstream to carriers.

In [ ]:
%%pyspark project.spark.compatibility
# Raw bronze rows where change_key_ats is the literal "\N" -- to take to upstream
(dlb
    .filter(F.col("change_key_ats") == "\\N")
    .show(5, truncate=False, vertical=True))

In [ ]:
%%pyspark project.spark.compatibility
(dlb
    .filter(F.col("change_key_ats") == "\\N")
    .groupBy("mno", "status")
    .count()
    .orderBy(F.desc("count"))
    .show(truncate=False))

In [ ]:
%%pyspark project.spark.compatibility
# Null breakdown per (mno, event_type) for the 4 hash-id columns.
# Useful for diagnosing WHICH carrier + WHICH event type drives the nulls.

def null_breakdown_by_event(df, col_name):
    return (
        df.groupBy("mno", "event_type")
          .agg(
              F.count("*").alias("total"),
              F.sum(F.col(col_name).isNull().cast("int")).alias(f"{col_name}_null"),
          )
          .withColumn("null_pct", F.round(F.col(f"{col_name}_null") / F.col("total"), 4))
          .orderBy(F.desc(f"{col_name}_null"))
    )

for column_name in ["imei", "imsi"]:
    print(f"=== {column_name} null breakdown by (mno, event_type) ===")
    null_breakdown_by_event(dlb_silver, column_name).show(truncate=False)

In [ ]:
%%pyspark project.spark.compatibility
# Bronze view: "null" includes actual null + literal "\N" + "MISSING_DATA"
# (these all become null after clean_str in silver)
NULL_LIKE = lambda c: F.col(c).isNull() | (F.col(c) == "\\N") | (F.col(c) == "MISSING_DATA")

print("=== bronze rows where imei is null/\\N/MISSING_DATA ===")
dlb.filter(NULL_LIKE("imei")).show(5, truncate=False, vertical=True)

print("=== bronze rows where imsi is null/\\N/MISSING_DATA ===")
dlb.filter(NULL_LIKE("imsi")).show(5, truncate=False, vertical=True)


In [ ]:
%%pyspark project.spark.compatibility
HEX64  = "(?i)^[0-9a-f]{64}$"
DIGITS = r"^\d+$"

non_hex = dlb_silver.filter(F.col("imei").isNotNull() & ~F.col("imei").rlike(HEX64))
print(f"=== imei non-hex64 total: {non_hex.count():,} ===\n")

print("--- DIGITS bucket (pure digits, top distinct values) ---")
(non_hex.filter(F.col("imei").rlike(DIGITS))
        .groupBy("imei", "mno").count()
        .orderBy(F.desc("count")).show(20, truncate=False))

print("--- OTHER bucket (everything else, top distinct values) ---")
(non_hex.filter(~F.col("imei").rlike(DIGITS))
        .groupBy("imei", "mno").count()
        .orderBy(F.desc("count")).show(20, truncate=False))


In [ ]:
%%pyspark project.spark.compatibility
HEX64  = "(?i)^[0-9a-f]{64}$"
DIGITS = r"^\d+$"

non_hex = dlb_silver.filter(F.col("imsi").isNotNull() & ~F.col("imsi").rlike(HEX64))
print(f"=== imsi non-hex64 total: {non_hex.count():,} ===\n")

print("--- DIGITS bucket (pure digits, top distinct values) ---")
(non_hex.filter(F.col("imsi").rlike(DIGITS))
        .groupBy("imsi", "mno").count()
        .orderBy(F.desc("count")).show(20, truncate=False))

print("--- OTHER bucket (everything else, top distinct values) ---")
(non_hex.filter(~F.col("imsi").rlike(DIGITS))
        .groupBy("imsi", "mno").count()
        .orderBy(F.desc("count")).show(20, truncate=False))



In [ ]:
%%pyspark project.spark.compatibility
# Show up to 5 example rows from BRONZE for each non-hex64 bucket; skip empty buckets.
# Bronze view = source-side column names + original `notes` JSON, useful for upstream.
HEX64  = "(?i)^[0-9a-f]{64}$"
DIGITS = r"^\d+$"

def _show_if_any(label, df, n=5):
    cnt = df.count()
    if cnt == 0:
        return
    print(f"=== {label} (total {cnt:,}, showing up to {n}) ===")
    df.show(n, truncate=False, vertical=True)

# Bronze filter: non-null, non-\N, non-MISSING_DATA, non-hex64
non_hex_imei = dlb.filter(
    F.col("imei").isNotNull()
    & (F.col("imei") != "\\N")
    & (F.col("imei") != "MISSING_DATA")
    & ~F.col("imei").rlike(HEX64)
)
non_hex_imsi = dlb.filter(
    F.col("imsi").isNotNull()
    & (F.col("imsi") != "\\N")
    & (F.col("imsi") != "MISSING_DATA")
    & ~F.col("imsi").rlike(HEX64)
)

_show_if_any("imei DIGITS bucket",
             non_hex_imei.filter(F.col("imei").rlike(DIGITS)))
_show_if_any("imei OTHER bucket",
             non_hex_imei.filter(~F.col("imei").rlike(DIGITS)))
_show_if_any("imsi DIGITS bucket",
             non_hex_imsi.filter(F.col("imsi").rlike(DIGITS)))
_show_if_any("imsi OTHER bucket",
             non_hex_imsi.filter(~F.col("imsi").rlike(DIGITS)))


In [ ]:
%%pyspark project.spark.compatibility
TAC8 = r"^\d{8}$"

non_8digit = dlb_silver.filter(F.col("tac").isNotNull() & ~F.col("tac").rlike(TAC8))
print(f"=== tac non-8-digit total: {non_8digit.count():,} ===\n")
non_8digit.show(20, truncate=False, vertical=True)


In [ ]:
%%pyspark project.spark.compatibility
# Bronze view of the tac=non-8-digit rows; pulls tac out of `notes` JSON on the fly
# so you can see the original notes string + source-side columns for upstream.
TAC8 = r"^\d{8}$"

bronze_tac_rows = (
    dlb
    .withColumn("_tac", F.get_json_object(F.col("notes"), "$.tac"))
    .filter(
        F.col("_tac").isNotNull()
        & ~F.col("_tac").isin("\\N", "", "MISSING_DATA")
        & ~F.col("_tac").rlike(TAC8)
    )
)

print(f"=== bronze rows with non-8-digit tac in notes: {bronze_tac_rows.count():,} ===\n")
bronze_tac_rows.show(20, truncate=False, vertical=True)


# Save Data

Publish the cleaned Silver to the partitioned Iceberg table, then verify partitioning.

In [ ]:
%%pyspark project.spark.compatibility
def write_iceberg(df, table_name, s3_location, schema_ddl, format_version="3"):
    """Create table if absent + overwrite all partitions covered by df.
    Idempotent for full-reload use case: re-running with the same data is a no-op
    because overwritePartitions atomically replaces each partition's data with df's.

    NOTE: if you change `schema_ddl`, drop the table manually first:
        spark.sql("DROP TABLE IF EXISTS <table_name>")
    """
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {table_name} (
            {schema_ddl}
        )
        USING iceberg
        PARTITIONED BY (event_date)
        LOCATION '{s3_location}'
        TBLPROPERTIES ('format-version' = '{format_version}')
    """)
    df.writeTo(table_name).overwritePartitions()


def verify_iceberg(table_name):
    """Run 4 sanity checks + row count."""
    print(f"=== A. SHOW CREATE TABLE ===")
    spark.sql(f"SHOW CREATE TABLE {table_name}").show(truncate=False)
    print(f"=== B. file paths ===")
    spark.sql(f"SELECT file_path FROM {table_name}.files LIMIT 5").show(truncate=False)
    print(f"=== C. file count ===")
    spark.sql(f"SELECT COUNT(*) AS file_count FROM {table_name}.files").show()
    print(f"=== D. partitions ===")
    spark.sql(f"""
        SELECT partition, spec_id, record_count
        FROM {table_name}.partitions ORDER BY partition LIMIT 10
    """).show(truncate=False)
    print(f"row count: {spark.table(table_name).count():,}")

In [ ]:
%%pyspark project.spark.compatibility
# Write dlb silver to Iceberg.
# Safe to rerun -- write_iceberg uses CREATE TABLE IF NOT EXISTS + overwritePartitions,
# so re-running with the same data replaces each partition's files (no duplication).
write_iceberg(
    df=dlb_silver,
    table_name="trust_score_v1_poc_silver.device_lookup_batch",
    s3_location="s3://enstream-lake-silver-dev/poc/tables/dataset=device_lookup_batch/version=1/",
    schema_ddl="""
        record_id BIGINT,
        phone_number_AC_hash STRING,
        phone_number_AT_hash STRING,
        mno STRING,
        imei STRING,
        imsi STRING,
        date DATE,
        event_timestamp TIMESTAMP,
        correlation_id STRING,
        related_id STRING,
        event_type STRING,
        tac STRING,
        oldIMSI STRING,
        oldIMEI STRING,
        fromADCSnapshot STRING,
        ingestion_ts TIMESTAMP,
        source_name STRING,
        source_event_id BIGINT,
        schema_version INT,
        event_date DATE
    """,
)

verify_iceberg("trust_score_v1_poc_silver.device_lookup_batch")

In [ ]:
%%pyspark project.spark.compatibility
# Inspect the partition folders Iceberg actually created in S3.
s3 = boto3.client('s3')
resp = s3.list_objects_v2(
    Bucket='enstream-lake-silver-dev',
    Prefix='poc/tables/dataset=device_lookup_batch/version=1/data/',
    Delimiter='/',         # key bit: list folders only, do not recurse
    MaxKeys=30
)

# the folder names S3 actually returns
for p in resp.get('CommonPrefixes', []):
    print(p['Prefix'])
